In [ ]:
import numpy as np
import pandas as pd
import scipy as scp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cmx
import dill
import pickle
import os
from tqdm import tqdm
from src.utils import init_hmp,GAMMA2,plot_hsmm,plot_val_hsmm,event_shape, sample_hmp
from mssmViz.plot import *
import mne
from mssm.models import *
from mssmViz.plot import *
from mssm.src.python.gamm_solvers import gd_coef_smooth,correct_coef_step_gen_smooth
from mssm.src.python.formula import build_model_matrix
from mssm.src.python.hsmm import HSMMFamily
from mssm.src.python.utils import computeAr1Chol
import dotenv
import arviz
dotenv.load_dotenv()

nc = 10
colors = [cmx.tab20(x) for x in np.linspace(0.0,min(0.1*nc,1.0),nc)]
single_size = 8/2.54

try:
    os.makedirs("./models")
except:
    print("Models directory already exists.")

try:
    os.makedirs("./models/callbacks/hsmm_LD")
except:
    print("Callbacks directory already exists.")

plt.rcParams["font.family"] = "Source Sans 3"
plt.rcParams["font.weight"] = "semibold"
plt.rcParams["font.size"] = 9
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["figure.titlesize"] = 12
math_font_size = 9
math_font = 'cm'

data_path = os.getenv("data_path")
results_path = os.getenv("results_path")
use_cores = int(os.getenv("n_cores"))
should_overwrite = bool(int(os.getenv("overwrite")))
print(should_overwrite)

## Data

In [ ]:
matdat = scp.io.loadmat(data_path + 'data4HMM_baseline_LD_excl_6_15_bs_100_.mat')
rt_dat = pd.read_csv(data_path + "rt_dat.csv")

In [ ]:
M = 10
n_events = 5
n_states = n_events + 1
sfreq = 100

# Create a montage and extract pca info for plotting
labels_obj = scp.io.loadmat(data_path + '/chanlocs.mat')
labels = [ch[0][0] for ch in labels_obj["chanlocs"][0]]

# Create a montage
montage = mne.channels.make_standard_montage('biosemi32')

# And then a largely empty info object to which we assign the montage.
info = mne.create_info(labels, sfreq=100, ch_types="eeg")
info.set_montage(montage, on_missing="ignore")
layout = mne.channels.make_eeg_layout(info)
pos = mne.channels.layout._find_topomap_coords(info, 'eeg')

pc_ev = matdat["latent10"][:M]
pc_w = matdat["coeff10"].T[:M,:].T
means = matdat["data"].mean(axis=0)

# Now get the data into the format required by mssm

pcs = [f"pc{pc}" for pc in range(1,M+1)]
dat = pd.DataFrame(matdat["normedscore10"][:,:M],columns=pcs)

# Now add the predictor variables of interest
dat["cond"] = matdat["conditionsB"]
dat["condS"] = "RS"
dat.loc[dat["cond"].values == 1,"condS"] = "W"
dat.loc[dat["cond"].values == 2,"condS"] = "PW"

dat["sub"] = matdat["subjects_varS"]
dat["trial"] = matdat["trials_varS"]

# Re-store trial variable
t_durs = matdat["y"] - matdat["x"] + 1
time = []
for t_dur in t_durs:
    time.extend(np.arange(t_dur[0])*10)

dat["time"] = time

# Create unique trial identifier!
dat["event"] = dat["sub"].astype(str) + "_" +  dat["trial"].astype(str)
rt_dat["event"] = rt_dat["subjects_var"].astype(str) + "_" + rt_dat["trials_var"].astype(str)
print(len(np.unique(dat["event"])),len(np.unique(rt_dat["event"])))

# Get frequency
evs, id, invev, ev_counts = np.unique(dat["event"],return_index=True,return_inverse=True,return_counts=True)
freq = []
for ev in evs:
    freq.append(rt_dat.loc[rt_dat["event"] == ev,"logGoogleFreq"].item())
dat["freq"] = np.array(freq)[invev]

# Exclude trials that have too short a duration.
min_dur = 5*(n_events) + n_states # 5 samples per bump + at least 1 per flat
excl_evs = evs[ev_counts < min_dur]

dat = dat.loc[~dat['event'].isin(excl_evs).values,:]
dat = dat.reset_index(drop=True)
rt_dat = rt_dat.loc[~rt_dat['event'].isin(excl_evs).values,:]
rt_dat = rt_dat.reset_index(drop=True)

# Assign correct data type and sort
dat = dat.astype({"sub":"O"})
rt_dat = rt_dat.astype({"subjects_var":"O"})

dat = dat.sort_values(['sub', 'trial', 'time'], ascending=[True, True, True])
dat = dat.reset_index(drop=True)
rt_dat = rt_dat.sort_values(['subjects_var', 'trials_var'], ascending=[True,True])
rt_dat = rt_dat.reset_index(drop=True)
print(len(np.unique(dat["event"])),len(np.unique(rt_dat["event"])))

# Build an event map for plotting
event_dict = {}
event_idx = 0
for ev in np.unique(dat["event"]):
    event_dict[ev] = event_idx
    event_idx += 1

In [ ]:
rt_dat

In [ ]:
dat

## Get HMP event

In [ ]:
suval, sid = np.unique(np.array(dat["event"]),return_index=True)
sid = np.sort(sid)

tuval, tid = np.unique(np.array(rt_dat["event"]),return_index=True)
tid = np.sort(tid)

template = event_shape(100,50)
template

## Fit original (shared) hmp model with varying gamma parameters and separate bump topologies per word type

In [ ]:
# Set up formulas
o_family = None # HMP template/bump is assumed observation model
d_family = GAMMA2(links=[LOG()]) # Gamma 2 for stage durations 
n_S = n_events + n_states
D = np.max(t_durs)
seed = 0

ys = []
Xs = []
form_n_coef = []
init_coef = []

build_mat_idx = []
obs_families = []
obs_formulas = []
build_matrix = []
links = []
pars = 0

# Emission models

event = 0
for j in range(n_S):
    obs_families.append([])

    # Model of mean of obs model in state j and of signal m
    for m in range(M):

        obs_families[-1].append(o_family)

        if j % 2 == 1:
            
            if event in [2,3]:
                obs_formulas.append(Formula(lhs(f"pc{m+1}"),[i(),l(["condS"])],data=dat))
                _ = build_penalties(obs_formulas[-1])

                if m == 0:
                    build_matrix.append(True)
                    Xs.append(build_model_matrix(obs_formulas[-1]))
                    build_mat_idx.append(len(build_mat_idx))
                else:
                    build_matrix.append(False)
                    Xs.append(None)
                    build_mat_idx.append(build_mat_idx[-1])
            else:
                obs_formulas.append(Formula(lhs(f"pc{m+1}"),[i()],data=dat))

                if len(build_matrix) == 0:
                    _ = build_penalties(obs_formulas[-1])
                    build_matrix.append(True)
                    Xs.append(build_model_matrix(obs_formulas[-1]))
                else:
                    build_matrix.append(False)
                    Xs.append(None)
                build_mat_idx.append(0)
            
            ys.append(dat[f"pc{m+1}"].values.reshape(-1,1))
            form_n_coef.append(obs_formulas[-1].n_coef)
            init_coef.extend(scp.stats.norm.rvs(size=obs_formulas[-1].n_coef,random_state=seed))
            seed += 1
            links.extend([Identity()])
            if m >= (M-1):
                event += 1
            pars += 1

# Duration models

d_dat = rt_dat
d_dat["d"] = np.arange(d_dat.shape[0])

d_families = []
d_formulas = []

d_build_mat_idx = len(build_mat_idx)

flat = 0
for j in range(n_S):

    if j % 2 == 1:
        d_families.append(None)
    else:

        d_families.append(d_family)
        # Model of mean of dur model in state j
        if flat in [3,4]:
            d_formulas.append(Formula(lhs("d"),[i(),l(["conditions"])],data=d_dat)) 

            _ = build_penalties(d_formulas[-1])
            Xs.append(build_model_matrix(d_formulas[-1]))
            build_matrix.append(True)
            build_mat_idx.append(len(build_mat_idx))

        else:
            d_formulas.append(Formula(lhs("d"),[i()],data=d_dat))

            if j == 0:
                _ = build_penalties(d_formulas[-1])
                Xs.append(build_model_matrix(d_formulas[-1]))
                build_matrix.append(True)
            else:
                Xs.append(None)
                build_matrix.append(False)
            
            build_mat_idx.append(d_build_mat_idx)
        
        form_n_coef.append(d_formulas[-1].n_coef)
        init_coef.extend([np.log(5) for _ in range(d_formulas[-1].n_coef)])
        ys.append(d_dat["d"].values.reshape(-1,1))

        links.extend(d_family.links)
        flat += 1
        pars += 1

# Initialize Family
T = np.diag(
    np.ones(n_S - 1), 1
)  # Super diagonal shift matrix for transitions

pi = np.zeros(n_S)  # Start in first state
pi[0] = 1

fam = HSMMFamily(pars,
                 links,
                 n_S,
                 obs_fams=obs_families,
                 d_fams=d_families,
                 sid = sid,
                 tid = tid,
                 D = D,
                 M = M,
                 starts_with_first=True,
                 ends_with_last=True,
                 ends_in_last=True,
                 T=T,
                 pi=pi,
                 scale = 1,
                 event_template=template,
                 hmp_fam="Gaussian",
                 n_cores=int(use_cores*2),
                 build_mat_idx=build_mat_idx)

# Initialize coef
coef_split_idx = form_n_coef[:-1]

for coef_i in range(1,len(coef_split_idx)):
    coef_split_idx[coef_i] += coef_split_idx[coef_i-1]

total_coef = np.sum(form_n_coef)
init_coef = np.array(init_coef).reshape(-1,1)

# Set up pseudo-penalty that is all zero
S_emb = scp.sparse.csc_array((total_coef,total_coef))

In [ ]:
# Estimate coefficients via quasi-Newton
path = results_path + "hsmm_LD_split.pickle"
if should_overwrite or (os.path.isfile(path) is False):

    # Start with some gradient descent steps
    conv_tol = 1e-7
    a = 1e-4
    coef = copy.deepcopy(init_coef)
    c_llk = fam.llk(coef,coef_split_idx,ys,Xs)

    for oi in tqdm(range(30)):

        grad = fam.gradient(coef,coef_split_idx,ys,Xs)

        next_coef = gd_coef_smooth(coef,grad,S_emb,a)

        # Prepare to check convergence
        prev_llk_cur_pen = c_llk - 0.5*coef.T@S_emb@coef

        # Perform step length control
        # Steps might throw warnings, return nan or inf, but mssm handles that
        with warnings.catch_warnings(): 
            warnings.simplefilter("ignore")
            coef,c_llk,c_pen_llk,a = correct_coef_step_gen_smooth(fam,ys,Xs,coef,next_coef,
                                                                  coef_split_idx,c_llk,S_emb,a)

        if np.abs(c_pen_llk - prev_llk_cur_pen) < conv_tol*np.abs(c_pen_llk):
            break
    
    # Now continue with quasi Newton
    llk_warp = lambda x: -1*fam.llk(x.reshape(-1,1),coef_split_idx,ys,Xs)
    grad_wrap = lambda x: -1*fam.gradient(x.reshape(-1,1),coef_split_idx,ys,Xs).flatten()
    opt = scp.optimize.minimize(llk_warp,
                                x0 = coef.flatten(),
                                jac=grad_wrap,
                                method = 'L-BFGS-B')

    
    coef = opt["x"].reshape(-1,1)
    print(opt)

    # Save results
    result = {"coef":coef,
              "opt":opt}

    with open(path, 'wb') as handle:
        pickle.dump(result, handle, protocol=pickle.HIGHEST_PROTOCOL)
else:
    with open(path, 'rb') as handle:
        result = pickle.load(handle)
        coef = result['coef']
        opt = result['opt']

split_coef = np.split(coef,coef_split_idx)

In [ ]:
# Plot overview for model
magnitudes = []
for bump in range(event):

    bump_coef = split_coef[bump*M:(bump+1)*M]
    bump_mat = Xs[build_mat_idx[bump*M]]
    
    if bump in [2,3]:
        lvl_magnitudes = []
        for lvl in ["RS","PW","W"]:
            obs = np.array([bump_mat[np.argmax(dat["condS"] == lvl),:]@bcoef for bcoef in bump_coef]).reshape(-1,1)
            lvl_magnitudes.append(obs*pc_ev)
        magnitudes.append(lvl_magnitudes)
    else:
        obs = np.array([bump_mat[np.argmax(dat["condS"] == "W"),:]@bcoef for bcoef in bump_coef]).reshape(-1,1)

        magnitudes.append(obs*pc_ev)

fig = plt.figure(figsize=(19 / 2.54, 6 / 2.54), layout="constrained")

plot_hsmm([fam,coef,coef_split_idx,Xs],ys,d_dat,
          network_graph=False,n_samples=10,
          series=3,fig=fig,plot_state_labels=False,
          magnitudes=magnitudes,
          pc_weights=pc_w,
          means=means,
          split_dur_by="conditions",
          split_dur_levels= ["RS", "PW", "W"],
          split_dur_levels_map = {"W":"Words", "PW":"Pseudo Words", "RS": "Random Strings"},
          positions=pos,topo_width=17.5,topo_height_rel=0.225,scale=0.01,d_dens_cutoff=0.01,
          emission_colors=[0.15,0.7,0.9],emission_label="PC"
          )

axs = fig.get_axes()
axs[1].set_xticks(np.round(axs[1].get_xticks(),decimals=0))
axs[2].set_ylim(-1.5,3.5)
axs[2].set_yticks([-1.5,0,1.5,2.5,3.5])

axs[0].set_xlabel("Time (samples)", fontweight="bold")
axs[1].set_xlabel("State Duration (samples)", fontweight="bold")
axs[3].set_xlabel("Time (samples)", fontweight="bold")

plt.savefig(results_path + "split_hmp_overview.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Plot forward pseudo residuals
for pc in range(M):
    fig = plt.figure(figsize=(19 / 2.54, 6 / 2.54), layout="constrained")

    plot_val_hsmm([fam,coef,coef_split_idx,Xs],ys,dat,d_dat,0,
                per_component=False,
                seed=5,predictive_examples=5,
                quantile_label_pos=0.5,wspace=0.01,fig=fig,n_samples=10,
                time_var="time",trials_var="event",select_component=pc,
                color_resids_by="sub",emission_label="PC"
                )

    axs = fig.get_axes()
    axs[1].set_ylim(-7,6)
    axs[1].axhline(0,color='black',linestyle='dashed')
    axs[2].set_ylim(-7,6)
    axs[2].axhline(0,color='black',linestyle='dashed')
    axs[3].set_ylim(-0.1,1)

    plt.savefig(results_path + f"split_hmp_res1_{pc}.png", format="png", bbox_inches='tight',dpi=300)
    if pc == 0:
        plt.show()
    plt.close()

### ar-1 process for Emissions

Pseudo-residuals show strong temporal correlations, so we include an ar-1 model of the
HMP deviations.

In [ ]:
# Specify parameter for ar-1 model.
rho = 0.7
Form = Formula(lhs(f"pc{1}"),[i()],data=dat,series_id="event")
Lrhoi,_ = computeAr1Chol(Form,rho)

In [ ]:
Lrhoi

In [ ]:
# Re-initialize Family with ar1-model
fam = HSMMFamily(pars,
                 links,
                 n_S,
                 obs_fams=obs_families,
                 d_fams=d_families,
                 sid = sid,
                 tid = tid,
                 D = D,
                 M = M,
                 starts_with_first=True,
                 ends_with_last=True,
                 ends_in_last=True,
                 T=T,
                 pi=pi,
                 scale = 1,
                 event_template=template,
                 hmp_fam="Gaussian",
                 n_cores=int(use_cores*2),
                 build_mat_idx=build_mat_idx,
                 Lrhoi=Lrhoi)

In [ ]:
# Again estimate coefficients via quasi-Newton
path = results_path + "hsmm_LD_split_ar.pickle"
if should_overwrite or (os.path.isfile(path) is False):

    conv_tol = 1e-7
    a = 1e-4
    c_llk = fam.llk(coef,coef_split_idx,ys,Xs)

    for oi in tqdm(range(30)):

        grad = fam.gradient(coef,coef_split_idx,ys,Xs)

        next_coef = gd_coef_smooth(coef,grad,S_emb,a)

        # Prepare to check convergence
        prev_llk_cur_pen = c_llk - 0.5*coef.T@S_emb@coef

        # Perform step length control
        # Steps might throw warnings, return nan or inf, but mssm handles that
        with warnings.catch_warnings(): 
            warnings.simplefilter("ignore")
            coef,c_llk,c_pen_llk,a = correct_coef_step_gen_smooth(fam,ys,Xs,coef,next_coef,
                                                                  coef_split_idx,c_llk,S_emb,a)

        if np.abs(c_pen_llk - prev_llk_cur_pen) < conv_tol*np.abs(c_pen_llk):
            break
    
    # Now continue with quasi Newton
    llk_warp = lambda x: -1*fam.llk(x.reshape(-1,1),coef_split_idx,ys,Xs)
    grad_wrap = lambda x: -1*fam.gradient(x.reshape(-1,1),coef_split_idx,ys,Xs).flatten()
    opt = scp.optimize.minimize(llk_warp,
                                x0 = coef.flatten(),
                                jac=grad_wrap,
                                method = 'L-BFGS-B')

    
    coef = opt["x"].reshape(-1,1)
    print(opt)

    # Save results
    result = {"coef":coef,
              "opt":opt}

    with open(path, 'wb') as handle:
        pickle.dump(result, handle, protocol=pickle.HIGHEST_PROTOCOL)
else:
    with open(path, 'rb') as handle:
        result = pickle.load(handle)
        coef = result['coef']
        opt = result['opt']

split_coef = np.split(coef,coef_split_idx)

In [ ]:
# Plot updated pseudo-residuals
for pc in range(M):
    fig = plt.figure(figsize=(19 / 2.54, 6 / 2.54), layout="constrained")

    plot_val_hsmm([fam,coef,coef_split_idx,Xs],ys,dat,d_dat,0,
                per_component=False,
                seed=5,predictive_examples=5,
                quantile_label_pos=0.5,wspace=0.01,fig=fig,n_samples=10,
                time_var="time",trials_var="event",select_component=pc,
                color_resids_by="sub",emission_label="PC"
                )

    axs = fig.get_axes()
    axs[1].set_ylim(-7,6)
    axs[1].axhline(0,color='black',linestyle='dashed')
    axs[2].set_ylim(-7,6)
    axs[2].axhline(0,color='black',linestyle='dashed')
    axs[3].set_ylim(-0.1,1)

    plt.savefig(results_path + f"split_hmp_ar_res1_{pc}.png",
                format="png", bbox_inches='tight',dpi=300)
    if pc == 0:
        plt.show()
    plt.close()

In [ ]:
# Plot average duration residuals to check whether model of state durations is appropriate.
def process_func(res_dat:pd.DataFrame):

    lvl_dats = []
    for lvl in ["PW", "RS", "W"]:
        lvl_freq = res_dat["logGoogleFreq"].values[res_dat["conditions"].values == lvl]
        lvl_freq = np.linspace(np.min(lvl_freq),np.max(lvl_freq),30)

        lvl_dat = pd.DataFrame({"logGoogleFreq":lvl_freq,
                                "conditions":[lvl for _ in range(30)],
                                "subjects_var":[res_dat["subjects_var"].values[0] for _ in range(30)],
                                "trials_var":[24 for _ in range(30)]})

        lvl_dats.append(lvl_dat)

    return "RdYlBu_r", lvl_dats, [0.1, 0.3, 0.8]

fig = plt.figure(figsize=(19 / 2.54, 19 / 2.54), layout="constrained")

resid_states = [4,6,8,10]

plot_val_hsmm([fam,coef,coef_split_idx,Xs],ys,dat,d_dat,4,
              per_component=False,fig=fig,
              seed=2,predictive_examples=5,
              quantile_label_pos=0.6,wspace=0.01,
              pred_vars=["logGoogleFreq"],resid_states=resid_states,
              time_var="time",trials_var="event",
              n_samples=100,resid_terms=[i(),l(["conditions"]),
                                         f(["logGoogleFreq"],by="conditions"),
                                         fs(["trials_var"],rf="subjects_var")],
              type_3_res="posterior_dur",hspace=0.05,plot_partial=3,which_pred=[0,1,2],
              process_dur_res_func=process_func,partial_title="Random Smooths",
              add_resid=False
            )

axs = fig.get_axes()
axs[4].set_ylabel("Duration Residuals", fontweight="bold",fontsize=plt.rcParams["axes.labelsize"])

for axi in range(0,4):
    axs[axi].set_xlabel("Word-likeness", fontweight="bold",fontsize=plt.rcParams["axes.labelsize"])
    axs[axi].set_title(f"Flat: {resid_states[axi]//2 + 1}\nWord-likeness Effects",
                       fontweight="bold",fontsize=plt.rcParams["axes.titlesize"])
for axi in range(4,8):
    axs[axi].set_xlabel("Trial", fontweight="bold",fontsize=plt.rcParams["axes.labelsize"])

for axi in range(12,16):
    axs[axi].set_ylim(-0.1,1.0)

fig.suptitle("Average Duration Residual Plots", fontweight="bold")

plt.savefig(results_path + "split_hmp_ar_res2.pdf", format="pdf", bbox_inches='tight')
plt.savefig(results_path + "split_hmp_ar_res2.png", format="png", bbox_inches='tight',dpi=300)
plt.show()

### Fitting a covariate dependent hierarchical HMP

In [ ]:
# We first decode the viterbi path of the simpler model to initialize the parameters of the more
# complex model
viterbi = fam.decode_viterbi(coef,coef_split_idx,ys,Xs)

In [ ]:
# Set up formulas
o_family = None
d_family = GAMMA2(links=[LOG()])
n_S = n_events + n_states
D = np.max(t_durs)
seed = 0

ys = []
Xs = []
form_n_coef = []

build_mat_idx = []
obs_families = []
shared_formulas = []
obs_formulas = []
build_matrix = []
links = []
pars = 0

event = 0
start_sep = len(build_matrix)
for j in range(n_S):
    obs_families.append([])

    # Model of mean of obs model in state j and of signal m
    # now includes random intercepts per subject
    for m in range(M):

        obs_families[-1].append(o_family)

        if j % 2 == 1:
            
            if event in [2,3]:
                obs_formulas.append(Formula(lhs(f"pc{m+1}"),[i(),l(["condS"]),ri("sub")],data=dat))
                _ = build_penalties(obs_formulas[-1])

                if m == 0:
                    build_matrix.append(True)
                    Xs.append((build_model_matrix(obs_formulas[-1])).tocsc())
                    build_mat_idx.append(len(build_mat_idx))
                else:
                    build_matrix.append(False)
                    Xs.append(None)
                    build_mat_idx.append(build_mat_idx[-1])
            else:
                obs_formulas.append(Formula(lhs(f"pc{m+1}"),[i(),ri("sub")],data=dat))
                _ = build_penalties(obs_formulas[-1])

                if len(build_matrix) == start_sep:
                    build_matrix.append(True)
                    Xs.append((build_model_matrix(obs_formulas[-1])).tocsc())
                else:
                    build_matrix.append(False)
                    Xs.append(None)

                build_mat_idx.append(start_sep)
            
            ys.append(dat[f"pc{m+1}"].values.reshape(-1,1))
            form_n_coef.append(obs_formulas[-1].n_coef)

            seed += 1
            links.extend([Identity()])
            if m >= (M-1):
                event += 1
            pars += 1

d_dat = rt_dat
d_dat["d"] = np.arange(d_dat.shape[0])

d_families = []
d_formulas = []

d_build_mat_idx = len(build_mat_idx)

flat = 0
for j in range(n_S):

    if j % 2 == 1:
        d_families.append(None)
    else:

        d_families.append(d_family)
        # Model of mean of dur model in state j - now includes smooth functions of word-likeness
        # and subject-specific ones of the experimental time-course.
        d_formulas.append(Formula(lhs("d"),[i(),
                                        l(["conditions"]),
                                        f(["logGoogleFreq"],by="conditions"),
                                        fs(["trials_var"],rf="subjects_var")],data=d_dat))

        if j == 0:
            _ = build_penalties(d_formulas[-1])
            Xs.append(build_model_matrix(d_formulas[-1]))
            build_matrix.append(True)
        else:
            Xs.append(None)
            build_matrix.append(False)
        
        build_mat_idx.append(d_build_mat_idx)
        form_n_coef.append(d_formulas[-1].n_coef)
        ys.append(d_dat["d"].values.reshape(-1,1))
        links.extend(d_family.links)
        flat += 1
        pars += 1

# Initialize Family
fam = HSMMFamily(pars,
                 links,
                 n_S,
                 obs_fams=obs_families,
                 d_fams=d_families,
                 sid = sid,
                 tid = tid,
                 D = D,
                 M = M,
                 starts_with_first=True,
                 ends_with_last=True,
                 ends_in_last=True,
                 T=T,
                 pi=pi,
                 scale = 1,
                 event_template=template,
                 hmp_fam="Gaussian",
                 n_cores=use_cores//2,
                 build_mat_idx=build_mat_idx,
                 Lrhoi=Lrhoi)

# Setup coef split idx - initialization is handled below
coef_split_idx = form_n_coef[:-1]

for coef_i in range(1,len(coef_split_idx)):
    coef_split_idx[coef_i] += coef_split_idx[coef_i-1]

total_coef = np.sum(form_n_coef)



In [ ]:
# Get initial estimates for coefficients and lambda parameters based on viterbi
# estimate of simpler hmp model states
with warnings.catch_warnings(): 
        warnings.simplefilter("ignore")
        init_coef, init_lambda = init_hmp(dat,d_dat,obs_formulas,d_formulas,
                                          [viterbi[s][1] for s in range(len(viterbi))],
                                          fam,Xs,ys,True,None,should_plot=True)

print(total_coef,init_coef.shape)

In [ ]:
# Sample hmp model via laplace approximate beam sampler.
def callback(iter, coef_samples, lam_samples):

    if iter % 50 == 0:
        state = {"coefs":coef_samples,
                 "lams":lam_samples}
        
        path = results_path + f"callbacks/hsmm_LD/rand_dur_state{iter}.pickle"
    
        with open(path, 'wb') as handle:
            pickle.dump(state, handle, protocol=pickle.HIGHEST_PROTOCOL)


path = results_path + "hsmm_LD_random_dur.dill"
if should_overwrite or (os.path.isfile(path) is False):
    coef_samples,lam_samples, hsmm = sample_hmp(fam,
                                                [init_coef + scp.stats.norm.rvs(size=len(init_coef),
                                                                             random_state=ins,scale=0.25).reshape(-1,1) for ins in range(2)],
                                                coef_split_idx,
                                                [*obs_formulas,*d_formulas],Xs,ys,
                                                n_burnin=625,n_iter=625,
                                                callback=callback,n_chains=2)
    
    result = {"coefs":coef_samples,
              "lams":lam_samples,
              "hsmm":hsmm}
        
    with open(path, 'wb') as handle:
        dill.dump(result, handle)

else:
    with open(path, 'rb') as handle:
        result = dill.load(handle)
        hsmm = result["hsmm"]

coef = hsmm.coef
coef_samples = result["coefs"]
lam_samples = result["lams"]
split_coef = np.split(coef,coef_split_idx)

In [ ]:
# Convergence checks
ess_all = []
rhat_all = []
mcse_all = []
par_samples_all = []
for samples in [coef_samples,np.log(lam_samples)]:
    n_chains = samples.shape[0]
    N = samples.shape[1]

    if N % 2 != 0:
        N -= 1

    par_samples = samples[:, :N, :]

    par_samples_split = np.zeros((2 * n_chains, N // 2, par_samples.shape[2]))

    split_idx = 0
    for chain in range(n_chains):
        par_samples_split[split_idx, :, :] = par_samples[chain, : N // 2, :]
        split_idx += 1
        par_samples_split[split_idx, :, :] = par_samples[chain, N // 2 :, :]
        split_idx += 1

    arviz_samples = arviz.convert_to_dataset(par_samples_split)
    ess = arviz.ess(arviz_samples, method="tail")
    rhat = arviz.rhat(arviz_samples)
    mcse = arviz.mcse(arviz_samples, method="mean")

    ess_all.append(ess.x.data)
    rhat_all.append(rhat.x.data)
    mcse_all.append(mcse.x.data)
    par_samples_all.append(par_samples_split)

# Coefficients
split_ess = np.split(ess_all[0],hsmm.coef_split_idx)
split_rhat = np.split(rhat_all[0],hsmm.coef_split_idx)
split_mcse = np.split(mcse_all[0],hsmm.coef_split_idx)
split_coef_samples = np.split(par_samples_all[0],hsmm.coef_split_idx,axis=2)
for fidx,form in enumerate(hsmm.formulas):

    sidx = form.get_smooth_term_idx()
    sidx = [idx for idx in sidx if isinstance(form.terms[idx],fs) is False]
    
    # Swap below to only check fixed effects and coefficients of smooth functions
    # with a non-trivial Kernel.
    use = None#[*form.get_linear_term_idx(),*sidx]

    _,fixedX,_ = hsmm.predict(use_terms=use,
                              n_dat=form.data,
                              par=fidx)

    fcidx = np.arange(fixedX.shape[1])[fixedX.sum(axis=0) > 0]
    coef_names = form.coef_names

    print(f"Summary Par. {fidx}:")

    if len(fcidx) > 1:
        print((
            f"Range ESS: {np.round(np.min(split_ess[fidx][fcidx]),decimals=2)} - "
            f"{np.round(np.max(split_ess[fidx][fcidx]),decimals=2)}, "
            f"Mean: {np.round(np.mean(split_ess[fidx][fcidx]),decimals=2)}, "
            f"SD: {np.round(np.std(split_ess[fidx][fcidx]),decimals=2)}"
            ))

        print((
            f"Range MCSE: {np.round(np.min(split_mcse[fidx][fcidx]),decimals=2)} - "
            f"{np.round(np.max(split_mcse[fidx][fcidx]),decimals=2)}, "
            f"Mean: {np.round(np.mean(split_mcse[fidx][fcidx]),decimals=2)}, "
            f"SD: {np.round(np.std(split_mcse[fidx][fcidx]),decimals=2)}"
            ))
        
        print((
            f"Range Rhat: {np.round(np.min(split_rhat[fidx][fcidx]),decimals=2)} - "
            f"{np.round(np.max(split_rhat[fidx][fcidx]),decimals=2)}, "
            f"Mean: {np.round(np.mean(split_rhat[fidx][fcidx]),decimals=2)}, "
            f"SD: {np.round(np.std(split_rhat[fidx][fcidx]),decimals=2)}"
            ))
    else:
        print((
            f"ESS: {np.round(np.mean(split_ess[fidx][fcidx]),decimals=2)}"
            ))

        print((
            f"MCSE: {np.round(np.mean(split_mcse[fidx][fcidx]),decimals=2)}"
            ))
        
        print((
            f"Rhat: {np.round(np.mean(split_rhat[fidx][fcidx]),decimals=2)}"
            ))

    for ci in fcidx:
        if split_rhat[fidx][ci] > 1.1 or split_ess[fidx][ci] < 10:
            print((
                "Warning: "
                f"ESS for coef {coef_names[ci]}: {np.round(split_ess[fidx][ci],decimals=2)}, "
                f"MCSE: {np.round(split_mcse[fidx][ci],decimals=2)}, "
                f"Rhat: {np.round(split_rhat[fidx][ci],decimals=2)}"
                ))
            
            ci_samples = split_coef_samples[fidx][:,:,ci]

            for chain_idx in range(ci_samples.shape[0]):
                plt.plot(ci_samples[chain_idx,:],label=f"Chain: {chain_idx}")
            plt.legend()
            plt.title(f"Trace for Coef: {coef_names[ci]} in model of Par.: {fidx}")
            plt.show()



print("Summary Lambda parameters:")

print((
        f"Range ESS: {np.round(np.min(ess_all[1]),decimals=2)} - "
        f"{np.round(np.max(ess_all[1]),decimals=2)}, "
        f"Mean: {np.round(np.mean(ess_all[1]),decimals=2)}, "
        f"SD: {np.round(np.std(ess_all[1]),decimals=2)}"
        ))

print((
        f"Range MCSE: {np.round(np.min(mcse_all[1]),decimals=2)} - "
        f"{np.round(np.max(mcse_all[1]),decimals=2)}, "
        f"Mean: {np.round(np.mean(mcse_all[1]),decimals=2)}, "
        f"SD: {np.round(np.std(mcse_all[1]),decimals=2)}"
        ))

print((
        f"Range Rhat: {np.round(np.min(rhat_all[1]),decimals=2)} - "
        f"{np.round(np.max(rhat_all[1]),decimals=2)}, "
        f"Mean: {np.round(np.mean(rhat_all[1]),decimals=2)}, "
        f"SD: {np.round(np.std(rhat_all[1]),decimals=2)}"
        ))


# Lambda parameters warnings
for ci in range(len(ess_all[1])):
    if rhat_all[1][ci] > 1.1 or ess_all[1][ci] < 10:
        print((
                "Warning: "
                f"ESS for lambda {ci}: {np.round(ess_all[1][ci],decimals=2)}, "
                f"MCSE: {np.round(mcse_all[1][ci],decimals=2)}, "
                f"Rhat: {np.round(rhat_all[1][ci],decimals=2)}"
                ))

        for chain_idx in range(par_samples_all[1].shape[0]):
            plt.plot(par_samples_all[1][chain_idx,:,ci],label=f"Chain: {chain_idx}")
        plt.legend()
        plt.title(f"Trace for lambda: {ci}")
        plt.show()


In [ ]:
# Marginal posterior of random intercepts for some subjects and some lambda parameters appears
# to be multimodal. Esepcially for subjects 1 and 18 the marginal seems to be multimodal for
# all intercepts. Socontinue sampling but with init coef set to averages across chains computed
# over different time-intervals.
re_sample_coef = coef_samples[:,[-1],:]

# Take averages from two intervals to initialize coef
re_sample_coef[0,0,:] = coef_samples[:,100:300,:].mean(axis=(0,1))
re_sample_coef[1,0,:] = coef_samples[:,400:600,:].mean(axis=(0,1))

# Now sample for only 50 iterations with no burn-in to check whether the estimates converge
# back to the previous modes.
def callback2(iter, coef_samples, lam_samples):

    if iter % 50 == 0:
        state = {"coefs":coef_samples,
                 "lams":lam_samples}
        
        path = results_path + f"callbacks/hsmm_LD/rand_dur_state{iter}_resampled.pickle"
    
        with open(path, 'wb') as handle:
            pickle.dump(state, handle, protocol=pickle.HIGHEST_PROTOCOL)


path = results_path + "hsmm_LD_random_dur_resampled.dill"
if should_overwrite or (os.path.isfile(path) is False):
    coef_resamples,lam_resamples, _ = sample_hmp(fam,
                                             [re_sample_coef[0,0,:].reshape(-1,1),
                                              re_sample_coef[1,0,:].reshape(-1,1)],
                                             coef_split_idx,
                                             [*obs_formulas,*d_formulas],Xs,ys,
                                             n_burnin=0,n_iter=50,
                                             callback=callback2,n_chains=2)
    
    result = {"coefs":coef_resamples,
              "lams":lam_resamples}
        
    with open(path, 'wb') as handle:
        dill.dump(result, handle)

else:
    with open(path, 'rb') as handle:
        result = dill.load(handle)
        coef_resamples = result["coefs"]
        lam_resamples = result["lams"]

In [ ]:
# Random intercepts immediately jump back to the previous modes. Suggesting that we got the
# important ones.
coef_names_total = []
for form in hsmm.formulas:
    coef_names_total.extend(form.coef_names)
coef_names_total = np.array(coef_names_total)

ri_idx = np.array([name in ["ri(sub_1)","ri(sub_18)"] for name in coef_names_total])

for ci in range(len(coef)):
    if ri_idx[ci]:
        for chain_idx in range(coef_samples.shape[0]):
            plt.plot(coef_samples[chain_idx,:coef_resamples.shape[1],ci],label=f"Chain: {chain_idx}")

        for chain_idx in range(coef_resamples.shape[0]):
            plt.plot(coef_resamples[chain_idx,:,ci],label=f"RE: Chain: {chain_idx}")
        plt.legend()
        plt.title(f"Trace for Coef: {coef_names_total[ci]}")
        plt.show()

In [ ]:
# Extract group-level / fixed model matrices with random effect columns zeroed out
fixed_Xs = []
for fidx in range(len(hsmm.formulas)):

    form = hsmm.formulas[fidx]
    
    if build_matrix[fidx]:
        
        sidx = form.get_smooth_term_idx()
        
        # Make sure to exclude random smooths
        sidx = [idx for idx in sidx if isinstance(form.terms[idx],fs) is False]

        _,fixedX,_ = hsmm.predict(use_terms=[*form.get_linear_term_idx(),
                                             *sidx],
                                             n_dat=form.data,
                                             par=fidx)
    
        fixed_Xs.append(fixedX)
    else:
        fixed_Xs.append(None)

In [ ]:
# Create subject condition interaction variable for easy visualization of state duration decoding
# for particular subjects
d_dat["sub_cond"] = d_dat["subjects_var"].astype("str") + "_" + d_dat["conditions"]

In [ ]:
# Plot overview for hierarchical model, including group level estimates of topographies
# and (a selection of) subject-level deviations.
magnitudes = []
for bump in range(event):

    bump_coef = copy.deepcopy(split_coef[bump*M:(bump+1)*M])
    bump_mat = fixed_Xs[build_mat_idx[bump*M]]
    
    if bump in [2,3]:
        lvl_magnitudes = []
        for lvl in ["RS","PW","W"]:
            obs = np.array([bump_mat[np.argmax(dat["condS"] == lvl),:]@bcoef for bcoef in bump_coef]).reshape(-1,1)
            lvl_magnitudes.append(obs*pc_ev)

        magnitudes.append(lvl_magnitudes)
    else:
        obs = np.array([bump_mat[np.argmax(dat["condS"] == "W"),:]@bcoef for bcoef in bump_coef]).reshape(-1,1)
        magnitudes.append(obs*pc_ev)
    
    
fig = plt.figure(figsize=(19 / 2.54, 6 / 2.54), layout="constrained")
axs = fig.subplots(1,3,gridspec_kw=dict(wspace=0.1))

dump_fig = plt.figure(figsize=(19 / 2.54, 6 / 2.54), layout="constrained")
dump_axs = dump_fig.subplots(1,2,gridspec_kw=dict(wspace=0.1))

plot_hsmm([fam,coef,coef_split_idx,fixed_Xs],ys,d_dat,
          network_graph=False,n_samples=10,
          series=3,plot_state_labels=False,
          magnitudes=magnitudes,
          pc_weights=pc_w,
          means=means,
          split_dur_by="conditions",
          split_dur_levels= ["RS", "PW", "W"],
          split_dur_levels_map = {"W":"Words (W)", "PW":"Pseudo Words", "RS": "Random Strings"},
          positions=pos,topo_width=17.5,topo_height_rel=0.225,scale=0.01,d_dens_cutoff=0.01,
          axs=[axs[0],*dump_axs],fig=fig
          )

axs[0].set_title("Group-level Estimates",fontweight="bold")
axs[0].set_xlabel("Time (samples)", fontweight="bold")

plot_hsmm([fam,coef,coef_split_idx,Xs],ys,d_dat,
          network_graph=False,n_samples=10,
          series=3,plot_state_labels=False,
          magnitudes=None,
          pc_weights=None,
          means=None,
          split_dur_by="sub_cond",
          split_dur_levels= ["7_RS", "7_PW", "7_W"],
          split_dur_levels_map = {"7_W":"Words", "7_PW":"Pseudo Words", "7_RS": "Random Strings"},
          positions=pos,topo_width=17.5,topo_height_rel=0.225,scale=0.01,d_dens_cutoff=0.01,
          axs=[axs[1],*dump_axs],fig=fig
          )

plt.close(dump_fig)
axs[1].tick_params(left=False, labelleft=False)
axs[1].set_title("States Single Subject",fontweight="bold")
axs[1].set_xlabel("Time (samples)", fontweight="bold")

# Subject-specific magnitudes plot
ax = axs[2]

magnitudes = []
do_plot = [0,1,2,3,4]
subjects = [19,2,18,5,4,13]


topo_width=2.05
topo_height_rel=0.17
x_spacing = 1.75
y_spacing = 3.5
total_events = 5

ax.set_xticks(np.linspace(1,x_spacing*total_events,total_events),
              ["1","2","3 (W)","4 (W)","5"])

x_lim = list(ax.get_xlim())
ax.set_ylim(0, y_spacing*len(subjects))
y_lim = list(ax.get_ylim())
y_lim[0] = 1
ax.set_yticks(np.linspace(*y_lim,len(subjects)),
              labels=subjects)

lvls = ax.get_xticks()

# Make sure top bumps don't reach over axis lim
ax.set_ylim(0,y_lim[-1] + ((topo_height_rel * np.diff(ax.get_ylim())[0]) / 2))

y_norm = np.diff(ax.get_ylim())[0]

y_bump_loc = (
    np.linspace(*y_lim,len(subjects)) - (topo_height_rel * 0.5 * y_norm)
) / y_norm

x_norm = np.diff(ax.get_xbound())[0]

topo_width_rel = topo_width / x_norm

x_bump_loc = (
    lvls - 0.5*(topo_width_rel*x_norm)
) / x_norm


for subi,sub in enumerate(subjects):
    
    fidx = 0
    bumpi = 0
    for bump in range(event):

        if bump in [2,3]:
            lvls = ["W"]
        else:
            lvls = [None]
        
        obs = [[] for _ in lvls]
        for m in range(M):

            for lvli,lvl in enumerate(lvls):

                pred_dat = pd.DataFrame({"condS":[lvl],
                                        "sub":[sub]})

                _,pred_mat, _ = hsmm.predict(None,n_dat=pred_dat,par=fidx)

                bump_coef = split_coef[fidx]
            
                pred = pred_mat@bump_coef
                obs[lvli].append(pred[0,0])

            fidx += 1
        
        obs = [np.array(lvl_obs).reshape(-1,1) * pc_ev for lvl_obs in obs]

        
        for lvl_obs in obs:

            if bump in do_plot:
                bump_axis = ax.inset_axes(
                    [
                        x_bump_loc[bumpi],
                        y_bump_loc[subi],  # 1,
                        topo_width_rel,
                        topo_height_rel,
                    ],
                    transform=ax.transAxes,
                )
                mne.viz.plot_topomap(
                    np.ndarray.flatten(lvl_obs.T @ pc_w.T),
                    pos,
                    cmap="Spectral_r",
                    axes=bump_axis,
                    show=False,
                    sensors=False,
                )
            bumpi += 1

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.set_ylabel("Subject", fontweight="bold")
ax.set_xlabel("Topography", fontweight="bold")
ax.set_title("Subject-level Topographies",fontweight="bold")

fig.align_xlabels()
fig.align_titles()

plt.savefig(results_path + "split_hmp_ar_rand_dur_overview.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Visualize all subject-level topographies
fig = plt.figure(figsize=(12 / 2.54, 22 / 2.54), layout="constrained")
ax = fig.subplots(1,1,gridspec_kw=dict(wspace=0.1))
magnitudes = []
do_plot = [0,1,2,3,4]
subjects = np.unique(dat["sub"])

state_cmp = matplotlib.colormaps["Paired"]

topo_width=1.3
topo_height_rel=0.1
x_spacing = 1.75
y_spacing = 3.5
total_events = 9

ax.set_xticks(np.linspace(1,x_spacing*total_events,total_events),
              ["1","2","3 (RS)","3 (PW)","3 (W)","4 (RS)","4 (PW)","4 (W)","5"])

x_lim = list(ax.get_xlim())
ax.set_ylim(0, y_spacing*len(subjects))
y_lim = list(ax.get_ylim())
y_lim[0] = 1
ax.set_yticks(np.linspace(*y_lim,len(subjects)),
              labels=subjects)

lvls = ax.get_xticks()

# Make sure top bumps don't reach over axis lim
ax.set_ylim(0,y_lim[-1] + ((topo_height_rel * np.diff(ax.get_ylim())[0]) / 2))

y_norm = np.diff(ax.get_ylim())[0]

y_bump_loc = (
    np.linspace(*y_lim,len(subjects)) - (topo_height_rel * 0.5 * y_norm)
) / y_norm

x_norm = np.diff(ax.get_xbound())[0]

topo_width_rel = topo_width / x_norm

x_bump_loc = (
    lvls - 0.5*(topo_width_rel*x_norm)
) / x_norm

xticks = ax.get_xticks()
yticks = ax.get_yticks()
x_lim = list(ax.get_xlim())
y_lim = list(ax.get_ylim())


ax.add_patch(patches.Rectangle([np.mean(xticks[[1,2]])+0.25,yticks[0]-2],height=yticks[-1]+3,
                               width=(np.mean(xticks[[4,5]]) - np.mean(xticks[[1,2]])) - 0.5,
                               facecolor=state_cmp(3),alpha=0.2,clip_on=False))

ax.add_patch(patches.Rectangle([np.mean(xticks[[4,5]])+0.25,yticks[0]-2],height=yticks[-1]+3,
                               width=(np.mean(xticks[[7,8]]) - np.mean(xticks[[4,5]]))-0.5,
                               facecolor=state_cmp(4),alpha=0.3,clip_on=False))

for subi,sub in enumerate(subjects):
    
    fidx = 0
    bumpi = 0
    for bump in range(event):

        if bump in [2,3]:
            lvls = ["RS","PW","W"]
        else:
            lvls = [None]
        
        obs = [[] for _ in lvls]
        for m in range(M):

            for lvli,lvl in enumerate(lvls):

                pred_dat = pd.DataFrame({"condS":[lvl],
                                        "sub":[sub]})

                _,pred_mat, _ = hsmm.predict(None,n_dat=pred_dat,par=fidx)

                bump_coef = split_coef[fidx]
            
                pred = pred_mat@bump_coef
                obs[lvli].append(pred[0,0])

            fidx += 1
        
        obs = [np.array(lvl_obs).reshape(-1,1) * pc_ev for lvl_obs in obs]

        
        for lvl_obs in obs:

            if bump in do_plot:
                bump_axis = ax.inset_axes(
                    [
                        x_bump_loc[bumpi],
                        y_bump_loc[subi],  # 1,
                        topo_width_rel,
                        topo_height_rel,
                    ],
                    transform=ax.transAxes,
                )
                mne.viz.plot_topomap(
                    np.ndarray.flatten(lvl_obs.T @ pc_w.T),
                    pos,
                    cmap="Spectral_r",
                    axes=bump_axis,
                    show=False,
                    sensors=False,
                )
            bumpi += 1

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.set_ylabel("Subject", fontweight="bold")
ax.set_xlabel("Topography", fontweight="bold")
ax.set_title("Subject-level Topographies",fontweight="bold")
plt.savefig(results_path + "all_subject_topographies_vertical.pdf",
            format="pdf", bbox_inches='tight')
plt.savefig(results_path + "all_subject_topographies_vertical.png",
            format="png", bbox_inches='tight',dpi=300)

plt.show()

In [ ]:
# Word-likeness effect estimates
fig = plt.figure(figsize=(19 / 2.54, 12 / 2.54), layout="constrained")
axs = fig.subplots(2, 3, gridspec_kw=dict(wspace=0.1,hspace=0.1))
axs = [axs[0,0],axs[0,1],axs[0,2],axs[1,0],axs[1,1],axs[1,2]]

d_pars = np.arange(50,56)
n_pred = 30

split_samples = np.split(
    np.concatenate([coef_samples[chain, :, :].T for chain in range(coef_samples.shape[0])], axis=1),
    hsmm.coef_split_idx,axis=0)
lvl_cols = [0.3, 0.1, 0.8]
lvl_cmp = matplotlib.colormaps["RdYlBu_r"]
lvl_map = {"W":"Words", "PW":"Pseudo Words", "RS": "Random Strings"}
handles = []
for axi,fidx in enumerate(d_pars):
    
    form = hsmm.formulas[fidx]
    ax = axs[axi]

    for lvli,lvl in enumerate(["RS", "PW", "W"]):

        lfreq = d_dat["logGoogleFreq"].values[d_dat["conditions"].values == lvl]

        freq = np.linspace(np.min(lfreq),np.max(lfreq),n_pred)

        lvl_dat = pd.DataFrame({"conditions":[lvl for _ in range(n_pred)],
                                "logGoogleFreq":freq,
                                "trials_var":[d_dat["trials_var"].values[0] for _ in range(n_pred)],
                                "subjects_var":[d_dat["subjects_var"].values[0] for _ in range(n_pred)]})

        sidx = form.get_smooth_term_idx()
        sidx = [idx for idx in sidx if isinstance(form.terms[idx],fs) is False]

        _,fixedXOffset,_ = hsmm.predict(use_terms=[*form.get_linear_term_idx()],
                                             n_dat=lvl_dat,
                                             par=fidx)
        
        _,fixedX,_ = hsmm.predict(use_terms=[*sidx],
                                             n_dat=lvl_dat,
                                             par=fidx)
        
        fidx_coef_samples = split_samples[fidx]
        smooth_samples = fixedX @ fidx_coef_samples
        offset_samples = fixedXOffset @ fidx_coef_samples
        

        handle = ax.plot(freq,np.quantile(smooth_samples,q=0.5,axis=1) + np.quantile(offset_samples,q=0.5,axis=1),
                         color=lvl_cmp(lvl_cols[lvli]),label=lvl_map[lvl])

        ax.fill([*freq,*np.flip(freq)],
                [*(np.quantile(smooth_samples,q=0.95,axis=1) + np.quantile(offset_samples,q=0.5,axis=1)),
                 *np.flip(np.quantile(smooth_samples,q=0.05,axis=1) + np.quantile(offset_samples,q=0.5,axis=1))],
                 color=lvl_cmp(lvl_cols[lvli]),
                 alpha=0.25)

        if axi == 0:
            handles.append(handle[0])

        ax.set_title(f"Flat: {axi+1}", fontweight="bold")
    
    if axi < 3:
        ax.set_ylim(0.85,1.6)
    else:
        ax.set_ylim(1,3)
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if axi >= 3:
        ax.set_xlabel("Word-likeness", fontweight="bold")
    if axi in [0,3]:
        ax.set_ylabel("$f($Word-likeness$)$", fontweight="bold")

fig.suptitle("Word-likeness Effects", fontweight="bold")
fig.legend(
        handles=handles,
        loc="lower center",
        ncol=3,
        bbox_transform=fig.transFigure,
        bbox_to_anchor=(0.5, -0.1),
        frameon=False,
    )

plt.savefig(results_path + "split_hmp_ar_rand_dur_wl_effect.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Credible intervals for differences between word types in effect of word-likeness
d_pars = np.arange(50,56)
n_pred = 30

split_samples = np.split(
    np.concatenate([coef_samples[chain, :, :].T for chain in range(coef_samples.shape[0])], axis=1),
    hsmm.coef_split_idx,axis=0)

lvl_cols = [0.3, 0.1, 0.8]
lvl_cmp = matplotlib.colormaps["RdYlBu_r"]

lvls = np.array(["RS", "PW", "W"])
lvl_idx = np.arange(3)

for ref_lvli in lvl_idx:

    ref_lvl = lvls[ref_lvli]

    fig = plt.figure(figsize=(19 / 2.54, 12 / 2.54), layout="constrained")
    axs = fig.subplots(2, 3, gridspec_kw=dict(wspace=0.1,hspace=0.1))
    axs = [axs[0,0],axs[0,1],axs[0,2],axs[1,0],axs[1,1],axs[1,2]]

    alt_lvls = np.array([idx for idx in lvl_idx if idx != ref_lvli])
    handles = []
    for axi,fidx in enumerate(d_pars):
        
        form = hsmm.formulas[fidx]
        ax = axs[axi]

        
        lfreq = d_dat["logGoogleFreq"].values[d_dat["conditions"].values == ref_lvl]

        freq = np.linspace(np.min(lfreq),np.max(lfreq),n_pred)

        lvl_dat = pd.DataFrame({"conditions":[ref_lvl for _ in range(n_pred)],
                                "logGoogleFreq":freq,
                                "trials_var":[d_dat["trials_var"].values[0] for _ in range(n_pred)],
                                "subjects_var":[d_dat["subjects_var"].values[0] for _ in range(n_pred)]})

        sidx = form.get_smooth_term_idx()
        sidx = [idx for idx in sidx if isinstance(form.terms[idx],fs) is False]


        _,fixedX_ref,_ = hsmm.predict(use_terms=[*form.get_linear_term_idx(),
                                            *sidx],
                                            n_dat=lvl_dat,
                                            par=fidx)
        
        fidx_coef_samples = split_samples[fidx]
        ref_samples = fixedX_ref @ fidx_coef_samples


        for lvli in alt_lvls:

            lvl = lvls[lvli]

            ref_freq = d_dat["logGoogleFreq"].values[d_dat["conditions"].values == lvl]

            lvl_dat = pd.DataFrame({"conditions":[lvl for _ in range(n_pred)],
                                    "logGoogleFreq":freq,
                                    "trials_var":[d_dat["trials_var"].values[0] for _ in range(n_pred)],
                                    "subjects_var":[d_dat["subjects_var"].values[0] for _ in range(n_pred)]})

            
            _,fixedX,_ = hsmm.predict(use_terms=[*form.get_linear_term_idx(),
                                                *sidx],
                                                n_dat=lvl_dat,
                                                par=fidx)
            
            alt_samples = fixedX @ fidx_coef_samples
            

            handle = ax.plot(freq[(freq >=  np.min(ref_freq)) & (freq <= np.max(ref_freq))],
                    np.quantile(alt_samples-ref_samples,q=0.5,axis=1)[(freq >=  np.min(ref_freq)) & (freq <= np.max(ref_freq))],
                    color=lvl_cmp(lvl_cols[lvli]),label=f"{lvl_map[lvl]} - {lvl_map[ref_lvl]}")

            
            ax.fill([*freq[(freq >=  np.min(ref_freq)) & (freq <= np.max(ref_freq))],
                     *np.flip(freq[(freq >=  np.min(ref_freq)) & (freq <= np.max(ref_freq))])],
                    [*(np.quantile(alt_samples-ref_samples,q=0.95,axis=1)[(freq >=  np.min(ref_freq)) & (freq <= np.max(ref_freq))]),
                     *np.flip(np.quantile(alt_samples-ref_samples,q=0.05,axis=1)[(freq >=  np.min(ref_freq)) & (freq <= np.max(ref_freq))])],
                    color=lvl_cmp(lvl_cols[lvli]),
                    alpha=0.25)

            if axi == 0:
                handles.append(handle[0])
            
        
        ax.axhline(0,color='black')
        ax.set_xlim(np.min(d_dat["logGoogleFreq"].values),np.max(d_dat["logGoogleFreq"].values))
        
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if axi >= 3:
                ax.set_xlabel("Word-likeness", fontweight="bold")
        if axi in [0,3]:
                ax.set_ylabel("Stage Duration Difference", fontweight="bold")


    fig.suptitle("Stage Duration Differences", fontweight="bold")
    fig.legend(
                handles=handles,
                loc="lower center",
                ncol=2,
                bbox_transform=fig.transFigure,
                bbox_to_anchor=(0.5, -0.1),
                frameon=False,
        )
    
    plt.savefig(results_path + f"split_hmp_ar_rand_dur_wl_diff_{ref_lvl}.pdf", format="pdf", bbox_inches='tight')
    plt.show()


In [ ]:
# Plot updated pseudo-residuals
for pc in range(M):
    fig = plt.figure(figsize=(19 / 2.54, 6 / 2.54), layout="constrained")

    plot_val_hsmm([fam,coef,coef_split_idx,Xs],ys,dat,d_dat,0,
                per_component=False,
                seed=5,predictive_examples=5,
                quantile_label_pos=0.5,wspace=0.01,fig=fig,n_samples=10,
                time_var="time",trials_var="event",select_component=pc,
                color_resids_by="sub",alpha=1,emission_label="PC"
                )

    axs = fig.get_axes()
    axs[1].set_ylim(-7,6)
    axs[1].axhline(0,color='black',linestyle='dashed')
    axs[2].set_ylim(-7,6)
    axs[2].axhline(0,color='black',linestyle='dashed')
    axs[3].set_ylim(-0.1,1)

    plt.savefig(results_path + f"split_hmp_ar_rand_dur_res1_{pc}.png",
                format="png", bbox_inches='tight',dpi=300)
    if pc == 0:
        plt.show()
    plt.close()

In [ ]:
# Plot updated average duration residuals
fig = plt.figure(figsize=(19 / 2.54, 19 / 2.54), layout="constrained")

resid_states = [4,6,8,10]
plot_val_hsmm([fam,coef,coef_split_idx,Xs],ys,dat,d_dat,4,
              per_component=False,fig=fig,
              seed=2,predictive_examples=5,
              quantile_label_pos=0.6,wspace=0.01,
              pred_vars=["logGoogleFreq"],resid_states=resid_states,
              time_var="time",trials_var="event",
              n_samples=100,resid_terms=[i(),l(["conditions"]),f(["logGoogleFreq"],by="conditions"),
                                         fs(["trials_var"],rf="subjects_var")],
              type_3_res="posterior_dur",hspace=0.05,plot_partial=3,which_pred=[0,1,2],
              process_dur_res_func=process_func,partial_title="Random Smooths",add_resid=False,
            )
axs = fig.get_axes()
axs[4].set_ylabel("Duration Residuals", fontweight="bold",fontsize=plt.rcParams["axes.labelsize"])

for axi in range(0,4):
    axs[axi].set_xlabel("Word-likeness", fontweight="bold",fontsize=plt.rcParams["axes.labelsize"])
    axs[axi].set_title(f"Flat: {resid_states[axi]//2 + 1}\nWord-likeness Effects",
                       fontweight="bold",fontsize=plt.rcParams["axes.titlesize"])
for axi in range(4,8):
    axs[axi].set_xlabel("Trial", fontweight="bold",fontsize=plt.rcParams["axes.labelsize"])

for axi in range(12,16):
    axs[axi].set_ylim(-0.1,1.0)

fig.suptitle("Average Duration Residual Plots", fontweight="bold")

plt.savefig(results_path + "split_hmp_ar_rand_dur_res2.pdf", format="pdf", bbox_inches='tight')
plt.savefig(results_path + "split_hmp_ar_rand_dur_res2.png", format="png", bbox_inches='tight',dpi=300)
plt.show()